# Thesis evidence audit

Independent review for `paper-astra`. This notebook executes the companion standard-library audit against original JSON exports, with an explicit eight-session cohort and a separate advisory run. It neither reads prior paper drafts nor executes pickle caches. It writes only its own audit outputs.

**Interpretation:** reproduction of a thesis number is not validation of a reading-benefit claim. Repeated derived events, protocol discrepancies, and unmatched geometric anchors are reported in `../review.md`. Source data is not changed.


In [1]:
from pathlib import Path
import contextlib
import importlib.util
import io
import json

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Experiments/data").is_dir())
script = root / "paper-astra/analysis/audit_evidence.py"
spec = importlib.util.spec_from_file_location("astra_audit", script)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
with contextlib.redirect_stdout(io.StringIO()):
    result = audit.main()
print(json.dumps(result["cohort"], indent=2))


{
  "participants": 4,
  "sessions": 8,
  "distinct_content_hashes": 2,
  "gaze_samples": 175860,
  "session_mean_hz": 89.88600952201071,
  "session_sd_hz": 0.2067790606383724,
  "session_mean_validity_pct": 95.55642202053994
}


## Cohort, materials, and recorded settings

Participant codes follow the existing four-person label order. The code does not publish names, demographic fields, or absolute session timestamps. Order is reconstructed from timestamps; the recorded order does not establish how assignment was decided.


In [2]:
for session in result["sessions"]:
    print({k: session[k] for k in ["participant", "condition", "text_title", "provider_metadata", "manual_interventions", "interventions", "boundaries", "quiz_answers"]})
print("Condition order:", result["condition_order"])


{'participant': 'P1', 'condition': 'with', 'text_title': 'Metal Detectors', 'provider_metadata': 'manual', 'manual_interventions': 5, 'interventions': 5, 'boundaries': {'immediate': 5}, 'quiz_answers': 3}
{'participant': 'P2', 'condition': 'with', 'text_title': 'Nuclear Science', 'provider_metadata': 'manual', 'manual_interventions': 7, 'interventions': 7, 'boundaries': {'immediate': 7}, 'quiz_answers': 5}
{'participant': 'P3', 'condition': 'with', 'text_title': 'Metal Detectors', 'provider_metadata': 'manual', 'manual_interventions': 3, 'interventions': 3, 'boundaries': {'immediate': 3}, 'quiz_answers': 3}
{'participant': 'P4', 'condition': 'with', 'text_title': 'Nuclear Science', 'provider_metadata': 'manual', 'manual_interventions': 4, 'interventions': 4, 'boundaries': {'immediate': 4}, 'quiz_answers': 5}
{'participant': 'P1', 'condition': 'without', 'text_title': 'Nuclear Science', 'provider_metadata': 'manual', 'manual_interventions': 9, 'interventions': 9, 'boundaries': {'immedia

## Event identity and behavioural sensitivity

Repeated payloads are counted within each session using exact serialised fixation/saccade payload equality. This detects extra copies independently of sequence IDs. The sensitivity recomputes the existing estimator after retaining one copy of each identical saccade payload. It is **not** a validated canonical event-identity rule.

The implemented estimator is in `post_events`: saccade start time, first forward movement within 30 seconds or the next intervention, pre/post windows capped at adjacent interventions, and a mean over non-empty event windows. Missing values are never converted to zero.


In [3]:
for row in result["integrity"]:
    print(row)
print("Original estimator:", json.dumps(result["behaviour"], indent=2))
print("Exact-payload deduplication sensitivity:", json.dumps(result["duplicate_payload_sensitivity"], indent=2))


{'participant': 'P1', 'condition': 'with', 'duplicate_gaze_timestamps': 0, 'nonincreasing_gaze_timestamps': 0, 'duplicate_gaze_sequence': 0, 'duplicate_intervention_ids': 0, 'duplicate_fixation_payloads': 38, 'duplicate_saccade_payloads': 78, 'context_unmatched_intervention_time': 0, 'telemetry_session_and_times_match': True}
{'participant': 'P2', 'condition': 'with', 'duplicate_gaze_timestamps': 0, 'nonincreasing_gaze_timestamps': 0, 'duplicate_gaze_sequence': 0, 'duplicate_intervention_ids': 0, 'duplicate_fixation_payloads': 38, 'duplicate_saccade_payloads': 88, 'context_unmatched_intervention_time': 0, 'telemetry_session_and_times_match': True}
{'participant': 'P3', 'condition': 'with', 'duplicate_gaze_timestamps': 0, 'nonincreasing_gaze_timestamps': 0, 'duplicate_gaze_sequence': 0, 'duplicate_intervention_ids': 0, 'duplicate_fixation_payloads': 0, 'duplicate_saccade_payloads': 29, 'context_unmatched_intervention_time': 0, 'telemetry_session_and_times_match': True}
{'participant': '

## Telemetry endpoints

Client RTT is five-second browser ping/pong telemetry. `pipeline-decision` uses the freshest backend ingest at dispatch, not a correlation-preserved trigger-to-display path. Provider RTT is conditional on matched replies. The advisory run is kept separate, including its client outlier.


In [4]:
print("Eight-session RTT:", json.dumps(result["cohort_rtt"], indent=2))
print("Separate advisory run:", json.dumps(result["advisory"], indent=2))


Eight-session RTT: {
  "researcher": {
    "n": 393,
    "p50": 1.0,
    "p95": 8.399999999999977,
    "p99": 11.159999999999968,
    "max": 28,
    "over_100ms_n": 0
  },
  "participant": {
    "n": 387,
    "p50": 1.0,
    "p95": 3.0,
    "p99": 10.139999999999986,
    "max": 41,
    "over_100ms_n": 0
  }
}
Separate advisory run: [
  {
    "duration_s": 91.19,
    "condition": {
      "conditionLabel": "Decision-maker advisory",
      "providerId": "external",
      "executionMode": "advisory"
    },
    "gaze_samples": 8004,
    "interventions": 3,
    "decision_proposal_records": 16,
    "unique_proposal_ids": 8,
    "proposal_record_statuses": {
      "pending": 8,
      "superseded": 6,
      "approved": 2
    },
    "latency": {
      "pipeline-decision": {
        "n": 14015,
        "p50": 6.0,
        "p95": 11.0,
        "p99": 11.0,
        "max": 46,
        "over_100ms_n": 0
      },
      "researcher": {
        "n": 18,
        "p50": 1.0,
        "p95": 18.049999999999

## Geometric trials

A trial key is intervention type and page index. Missing word measurements are excluded from measured pairs and reported explicitly. Over-repositioning uses the table analyser threshold, ON > OFF + 1 px. Cross-version matching checks the recorded token identity and the supposed restore-independent OFF displacement. This reruns aggregation, not the historical browser experiment.


In [5]:
print(json.dumps(result["sweep"], indent=2))
print("Legacy discovery issue:", result["legacy_loader"])
print("Data inputs with SHA-256 hashes:", len(result["input_hashes"]))
print("Audit checkout:", result["review_commit"])


{
  "original": {
    "scheduled_trials": 66,
    "measurable_pairs": 62,
    "missing_pairs": 4,
    "off_observed": 62,
    "on_observed": 64,
    "off_median_px": 32.39,
    "on_median_px": 104.38499999999999,
    "over_reposition_n": 45,
    "threshold_px": 1,
    "missing_rows": [
      {
        "intervention": "Line width -120px (680->560)",
        "pageIndex": 3,
        "offDisplacementPx": null,
        "onDisplacementPx": 357.08
      },
      {
        "intervention": "Line width -120px (680->560)",
        "pageIndex": 4,
        "offDisplacementPx": null,
        "onDisplacementPx": null
      },
      {
        "intervention": "Line width -120px (680->560)",
        "pageIndex": 5,
        "offDisplacementPx": null,
        "onDisplacementPx": null
      },
      {
        "intervention": "Line width +80px (680->760)",
        "pageIndex": 4,
        "offDisplacementPx": null,
        "onDisplacementPx": 265.13
      }
    ]
  },
  "revised": {
    "scheduled_trials": 6

## Scope of validation

The script checks schema and matching telemetry/session IDs and times; reproduces the original behavioural CSV summaries within numeric tolerance; checks gaze ordering, sequence/ID uniqueness and repeated derived payloads; and compares sweep identities and missingness. It does not validate event classification against human annotation, infer an intervention effect, recreate display timing, prove participant consent, or retest real hardware.

The separate backend test command and its 101/101 result are documented in `../code-map.md`; they were not run by this notebook.
